# 01 – Aineiston lataus: täysistuntopuheet 2015–2026

**Tehtävä:** Eduskunnan avoimen datan puhe-exportti (NDJSON) muutetaan taulukoksi, jossa on yksi rivi per puheenvuoro. Tekstiä ei muuteta; siivous tehdään muistiossa `02_siivous`.

**Syöte:** `Data/dataset-*.ndjson` (ladattu 23.9.2026; tiedostoa ei muuteta)
**Tulos:** `Data/processed/speeches_raw.parquet`

## Aineisto

Eduskunnan avoin data, lisenssi CC BY 4.0. Jokainen rivi on JSON-olio, jonka kentässä `puheenvuoro` ovat puheen tiedot. Simolan ym. aineisto on OCR-tunnistettu PDF-pöytäkirjoista; tämä aineisto on alun perin digitaalista. Puhujan tunniste ja eduskuntaryhmä ovat valmiina jokaisessa puheessa, joten nimiä ei tarvitse yhdistää sumeasti eikä OCR-virheitä korjata.

In [1]:
import glob
import json

import pandas as pd

LAHTEET = sorted(glob.glob("../Data/dataset-*.ndjson"))
TULOS = "../Data/processed/speeches_raw.parquet"
print(LAHTEET)

['../Data/dataset-0179e17f-35c4-4439-b88a-b4b2153e72e7.ndjson']


### Yhden rivin rakenne

Ensimmäisestä rivistä nähdään, missä kentissä tarvittavat tiedot ovat.

In [2]:
with open(LAHTEET[0], encoding="utf-8") as tiedosto:
    ensimmainen = json.loads(tiedosto.readline())

puhe = ensimmainen["puheenvuoro"]
print("Kentät:", list(puhe.keys()))
print()
print("Puhuja:", puhe["puhuja"])
print("Tyyppi:", puhe["puheenvuorotyyppinimi"])
print("Asia:", puhe["asia"]["fi"])
print("Alku:", puhe["aloitushetki"])
print()
print("Teksti (alku):", puhe["puheenvuoro"][:300])
print()
print("Puhemiehen repliikki:", puhe["puheenjohtajanRepliikki"])

Kentät: ['id', 'tunnus', 'tila', 'puheenvuorotyyppikoodi', 'puheenvuorotyyppinimi', 'valtiopaiva', 'valtiopaivavuosi', 'taysistuntonumero', 'asia', 'poytakirjanasiankohta', 'asiakirjaviitteet', 'puhuja', 'aloitushetki', 'lopetushetki', 'kellonaika', 'puheenvuoro', 'puheenvuoroXml', 'puheenjohtajanRepliikki', 'historiallinen']

Puhuja: {'henkilonro': '971', 'asema': None, 'etunimi': 'Johanna', 'sukunimi': 'Ojala-Niemelä', 'lisatieto': 'sd', 'eduskuntaryhma_tunnus': 'SD01~SOSIALIDEMOKRAATTINEN EDUSKUNTARYHMÄ'}
Tyyppi: Vastauspuheenvuoro
Asia: {'eduskuntatunnus': 'HE 175/2026 vp', 'nimeketeksti': 'Hallituksen esitys eduskunnalle valtion talousarvioksi vuodelle 2027'}
Alku: 2026-09-23T07:52:16+00:00

Teksti (alku): Arvoisa herra puhemies! Meillä on tällä hetkellä Suomessa Euroopan korkein työttömyys. On hyvä, että valoa on vihdoin tunnelin päässä, mutta kyllä nämä kasvuluvut ovat vielä aika vaatimattomia. Niillä ei kovin paljon kannata kyllä rehvastella. Edustaja Viitanen toivoi, että Lex 

Huomioita:
- **Puhemiehen repliikit** ("Kiitoksia. — Edustaja Kauma, olkaa hyvä.") ovat omassa kentässään, eivät omina puheinaan. Simola ym. poistavat puhemiehen puheet; tässä aineistossa ne ovat valmiiksi erillään.
- **`eduskuntaryhma_tunnus`** on puhujan eduskuntaryhmä *puheen hetkellä*, joten ryhmää vaihtaneiden edustajien ryhmä on jokaisessa puheessa oikein.
- **`asema`** on täytetty, kun puhuja puhuu ministerinä (esim. "Pääministeri").
- Tekstissä on pikakirjoittajan merkintöjä hakasulkeissa, esim. `[Puhemies koputtaa]` ja `[Välihuutoja]`. Ne poistetaan siivouksessa.

### Sarakkeet

Sarakkeiden nimet ovat englanniksi, koska niitä käytetään koodissa.

| sarake | alkuperäinen kenttä | sisältö |
|---|---|---|
| `speech_id` | `id` | puheenvuoron tunnus, esim. PUH 86/2026/2/1/34 |
| `record_id` | `tunnus.fi` | pöytäkirja, esim. PTK 86/2026 vp |
| `session_year` | `valtiopaivavuosi` | valtiopäivävuosi |
| `start_time`, `end_time` | `aloitushetki`, `lopetushetki` | puheen alku ja loppu (UTC) |
| `speech_type_code` | `puheenvuorotyyppikoodi` | T = varsinainen, V = vastaus, N = nopeatahtinen, E = esittely, R = ryhmä |
| `item_id`, `item_title` | `asia.fi` | käsiteltävä asia, esim. HE 175/2026 vp |
| `person_id` | `puhuja.henkilonro` | edustajan pysyvä tunnus |
| `first_name`, `last_name` | `puhuja.etunimi`, `sukunimi` | nimi |
| `role_fi` | `puhuja.asema` | ministerin tehtävä, muuten tyhjä |
| `group_code`, `group_name_fi` | `puhuja.eduskuntaryhma_tunnus` | esim. SD01 ja SOSIALIDEMOKRAATTINEN EDUSKUNTARYHMÄ |
| `chair_name` | `puheenjohtajanRepliikki.puheenjohtaja` | istuntoa johtanut puhemies |
| `text` | `puheenvuoro` | puheen teksti sellaisenaan |

In [3]:
def hae(sanakirja, *avaimet):
    """Hakee sisäkkäisen kentän; palauttaa None, jos jokin taso puuttuu."""
    for avain in avaimet:
        if sanakirja is None:
            return None
        sanakirja = sanakirja.get(avain)
    return sanakirja


def lue_rivi(rivi):
    """Muuttaa yhden JSON-rivin taulukon riviksi (sanakirjaksi)."""
    tieto = json.loads(rivi)
    p = tieto.get("puheenvuoro") or {}
    puhuja = p.get("puhuja") or {}
    ryhma = puhuja.get("eduskuntaryhma_tunnus") or ""
    ryhman_koodi, _, ryhman_nimi = ryhma.partition("~")
    puheenjohtaja = p.get("puheenjohtajanRepliikki") or {}
    return {
        "speech_id": p.get("id"),
        "record_id": hae(p, "tunnus", "fi"),
        "session_year": p.get("valtiopaivavuosi"),
        "sitting_no": p.get("taysistuntonumero"),
        "start_time": p.get("aloitushetki"),
        "end_time": p.get("lopetushetki"),
        "speech_type_code": (p.get("puheenvuorotyyppikoodi") or "").strip(),
        "speech_type_fi": (p.get("puheenvuorotyyppinimi") or "").strip(),
        "item_id": hae(p, "asia", "fi", "eduskuntatunnus"),
        "item_title": hae(p, "asia", "fi", "nimeketeksti"),
        "agenda_item_title": hae(p, "poytakirjanasiankohta", "fi", "nimeketeksti"),
        "person_id": puhuja.get("henkilonro"),
        "first_name": puhuja.get("etunimi"),
        "last_name": puhuja.get("sukunimi"),
        "role_fi": puhuja.get("asema"),
        "party_label": puhuja.get("lisatieto"),
        "group_code": ryhman_koodi or None,
        "group_name_fi": ryhman_nimi or None,
        "status": p.get("tila"),
        "chair_name": puheenjohtaja.get("puheenjohtaja"),
        "text": p.get("puheenvuoro") or "",
    }

In [4]:
rivit = []
for polku in LAHTEET:
    with open(polku, encoding="utf-8") as tiedosto:
        for rivi in tiedosto:
            if rivi.strip():
                rivit.append(lue_rivi(rivi))

puheet = pd.DataFrame(rivit)
puheet["start_time"] = pd.to_datetime(puheet["start_time"], utc=True)
puheet["end_time"] = pd.to_datetime(puheet["end_time"], utc=True)
puheet = puheet.sort_values("start_time").reset_index(drop=True)
print("Puheita:", len(puheet))

Puheita: 133648


## Tarkistukset

- Onko sama puhe mukana kahdesti?
- Kuinka monessa puheessa ei ole tekstiä?
- Mikä on aineiston aikaväli?
- Ovatko vuosittaiset määrät järkeviä? Vaalivuosina (2015, 2019, 2023) puheita on vähemmän, koska vaalien ympärillä täysistuntoja on vähemmän.

In [5]:
print("Päällekkäisiä tunnuksia:", puheet["speech_id"].duplicated().sum())
print("Tyhjiä puheita:", (puheet["text"].str.strip() == "").sum())
print("Ensimmäinen puhe:", puheet["start_time"].min())
print("Viimeinen puhe:", puheet["start_time"].max())
print("Eri puhujia:", puheet["person_id"].nunique())

Päällekkäisiä tunnuksia: 0
Tyhjiä puheita: 10
Ensimmäinen puhe: 2015-05-12 11:03:46+00:00
Viimeinen puhe: 2026-09-23 07:52:16+00:00
Eri puhujia: 373


In [6]:
puheet["session_year"].value_counts().sort_index()

session_year
2015     8472
2016    13649
2017    13235
2018    15277
2019     6900
2020    12355
2021    12211
2022    12736
2023     6562
2024    12587
2025    11983
2026     7681
Name: count, dtype: int64

In [7]:
print("Puheenvuorotyypit:")
puheet["speech_type_fi"].replace("", "(tyhjä)").value_counts()

Puheenvuorotyypit:


speech_type_fi
Vastauspuheenvuoro           62047
Varsinainen puheenvuoro      55587
Nopeatahtinen puheenvuoro     5727
(tyhjä)                       4319
Esittelypuheenvuoro           3963
Ryhmäpuheenvuoro              2005
Name: count, dtype: int64

Noin 4 300 puheelta puuttuu tyyppi. Niitä on joka vuonna, ja noin 90 % on ministerien puheita (tarkistus alla). Puheet pidetään aineistossa tyypillä *määrittelemätön*.

In [8]:
print("Eduskuntaryhmät:")
puheet["group_name_fi"].value_counts(dropna=False)

Eduskuntaryhmät:


group_name_fi
SOSIALIDEMOKRAATTINEN EDUSKUNTARYHMÄ       28768
PERUSSUOMALAISTEN EDUSKUNTARYHMÄ           25113
KANSALLISEN KOKOOMUKSEN EDUSKUNTARYHMÄ     24033
KESKUSTAN EDUSKUNTARYHMÄ                   24003
VIHREÄ EDUSKUNTARYHMÄ                      10295
VASEMMISTOLIITON EDUSKUNTARYHMÄ             9359
KRISTILLISDEMOKRAATTINEN EDUSKUNTARYHMÄ     5178
RUOTSALAINEN EDUSKUNTARYHMÄ                 4358
SININEN EDUSKUNTARYHMÄ                      1680
LIIKE NYT -EDUSKUNTARYHMÄ                    414
EDUSKUNTARYHMÄ VALTA KUULUU KANSALLE         185
None                                          80
EDUSKUNTARYHMÄ TIMO VORNANEN                  62
EDUSKUNTARYHMÄÄN KUULUMATON                   55
TÄHTILIIKKEEN EDUSKUNTARYHMÄ                  38
EDUSKUNTARYHMÄ WILLE RYDMAN                   12
EDUSKUNTARYHMÄ JUVONEN                         8
UUSI VAIHTOEHTO EDUSKUNTARYHMÄ                 4
EDUSTAJA VÄYRYNEN                              3
Name: count, dtype: int64

In [9]:
tyypiton = puheet[puheet["speech_type_fi"] == ""]
print("Tyypittömiä puheita vuosittain:")
print(tyypiton["session_year"].value_counts().sort_index().to_dict())
print(f"Näistä ministerin puheita: {tyypiton['role_fi'].notna().mean():.0%}")

Tyypittömiä puheita vuosittain:
{'2015': 223, '2016': 382, '2017': 445, '2018': 459, '2019': 271, '2020': 473, '2021': 482, '2022': 440, '2023': 162, '2024': 384, '2025': 327, '2026': 271}
Näistä ministerin puheita: 90%


Pienet ryhmät (Liike Nyt, Valta kuuluu kansalle, yhden hengen ryhmät) ja 80 puhetta ilman ryhmää rajataan pois muistiossa 02.

## Tallennus

In [10]:
puheet.to_parquet(TULOS, index=False)
print("Tallennettu", TULOS, "–", len(puheet), "puhetta")

Tallennettu ../Data/processed/speeches_raw.parquet – 133648 puhetta
